# 🎬 Movie Recommendation System — Big Data + PySpark

**Full Pipeline: Data Ingestion → EDA → ALS Collaborative Filtering → Content-Based → Hybrid Model → Dashboard**

---
### 📋 Steps in this Notebook
| Step | Description |
|------|-------------|
| **1** | Install dependencies |
| **2** | Download MovieLens dataset |
| **3** | Initialize PySpark session |
| **4** | Load & explore data (EDA) |
| **5** | Feature engineering |
| **6** | ALS Collaborative Filtering |
| **7** | Content-Based Filtering (TF-IDF) |
| **8** | Hybrid Ensemble Model |
| **9** | Evaluation metrics |
| **10** | Interactive Dashboard (Plotly) |

> ✅ **Tested on Google Colab & local Jupyter with Python 3.9+**

---
## ⚙️ Step 1 — Install Dependencies

In [2]:
# Run once — restart kernel after if needed
import sys


!python -m pip install pyspark==3.4.1 --quiet
!python -m pip install pandas numpy matplotlib seaborn plotly --quiet
!python -m pip install scikit-learn requests tqdm ipywidgets --quiet
# Verify PySpark
import pyspark
print(f'✅ PySpark version: {pyspark.__version__}')
print(f'✅ Python version:  {sys.version}')

FileNotFoundError: [WinError 2] The system cannot find the file specified

---
## 📥 Step 2 — Download MovieLens Dataset

In [3]:
import os, zipfile, urllib.request
from pathlib import Path

DATA_DIR = Path('data/ml-latest-small')
ZIP_PATH = Path('data/ml-latest-small.zip')
URL = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'

os.makedirs('data', exist_ok=True)

if not DATA_DIR.exists():
    print('⬇️  Downloading MovieLens Small (100K ratings)...')
    urllib.request.urlretrieve(URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('data')
    print('✅ Dataset downloaded and extracted!')
else:
    print('✅ Dataset already exists — skipping download')

# Show files
for f in DATA_DIR.iterdir():
    size = f.stat().st_size / 1024
    print(f'   📄 {f.name:<25} ({size:,.1f} KB)')

⬇️  Downloading MovieLens Small (100K ratings)...
✅ Dataset downloaded and extracted!
   📄 links.csv                 (193.3 KB)
   📄 movies.csv                (482.8 KB)
   📄 ratings.csv               (2,425.5 KB)
   📄 README.txt                (8.1 KB)
   📄 tags.csv                  (115.9 KB)


---
## 🔥 Step 3 — Initialize PySpark Session

In [4]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName('MovieRecommender')
    .master('local[*]')                          # Use all CPU cores
    .config('spark.driver.memory', '2g')
    .config('spark.executor.memory', '2g')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '8') # Small dataset → fewer partitions
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')  # Suppress verbose logs

print(f'✅ Spark {spark.version} started')
print(f'   Master: {spark.sparkContext.master}')
print(f'   App:    {spark.sparkContext.appName}')
print(f'   UI:     http://localhost:4040')

ModuleNotFoundError: No module named 'pyspark'

---
## 📊 Step 4 — Load Data & Exploratory Data Analysis

In [5]:
# ── Load all CSVs into Spark DataFrames ───────────────────────────────────
BASE = 'data/ml-latest-small'

ratings_df = spark.read.csv(f'{BASE}/ratings.csv',  header=True, inferSchema=True)
movies_df  = spark.read.csv(f'{BASE}/movies.csv',   header=True, inferSchema=True)
tags_df    = spark.read.csv(f'{BASE}/tags.csv',      header=True, inferSchema=True)
links_df   = spark.read.csv(f'{BASE}/links.csv',     header=True, inferSchema=True)

print('📊 Dataset Summary')
print('='*40)
print(f'  Ratings:  {ratings_df.count():>10,}')
print(f'  Movies:   {movies_df.count():>10,}')
print(f'  Tags:     {tags_df.count():>10,}')
print(f'  Users:    {ratings_df.select("userId").distinct().count():>10,}')

print('\n📌 Ratings Schema:')
ratings_df.printSchema()
print('\n📌 Sample Ratings:')
ratings_df.show(5)

NameError: name 'spark' is not defined

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Convert aggregates to Pandas for plotting
rating_dist  = ratings_df.groupBy('rating').count().orderBy('rating').toPandas()
user_activity = (ratings_df.groupBy('userId')
                  .count().withColumnRenamed('count','num_ratings')
                  .toPandas())
movie_pop    = (ratings_df.groupBy('movieId')
                  .agg(F.count('rating').alias('num_ratings'),
                       F.avg('rating').alias('avg_rating'))
                  .toPandas())

fig = plt.figure(figsize=(16, 10))
fig.suptitle('MovieLens — Exploratory Data Analysis', fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Rating distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.bar(rating_dist['rating'], rating_dist['count'],
        color=sns.color_palette('viridis', len(rating_dist)))
ax1.set_xlabel('Rating'); ax1.set_ylabel('Count')
ax1.set_title('Rating Distribution'); ax1.grid(axis='y', alpha=0.3)

# 2. User activity histogram
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(user_activity['num_ratings'], bins=50, color='steelblue', edgecolor='white')
ax2.set_xlabel('Ratings per user'); ax2.set_ylabel('Users')
ax2.set_title('User Activity Distribution'); ax2.grid(axis='y', alpha=0.3)

# 3. Movie popularity histogram
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(movie_pop['num_ratings'], bins=50, color='coral', edgecolor='white')
ax3.set_xlabel('Ratings per movie'); ax3.set_ylabel('Movies')
ax3.set_title('Movie Popularity Distribution'); ax3.grid(axis='y', alpha=0.3)

# 4. Average rating vs. popularity scatter
ax4 = fig.add_subplot(gs[1, 0:2])
sample = movie_pop.sample(min(1000, len(movie_pop)), random_state=42)
sc = ax4.scatter(sample['num_ratings'], sample['avg_rating'],
                 alpha=0.4, c=sample['avg_rating'],
                 cmap='RdYlGn', s=20)
plt.colorbar(sc, ax=ax4, label='Avg Rating')
ax4.set_xlabel('Number of ratings'); ax4.set_ylabel('Average rating')
ax4.set_title('Popularity vs. Average Rating'); ax4.grid(alpha=0.2)

# 5. Genre distribution
ax5 = fig.add_subplot(gs[1, 2])
from collections import Counter
genre_counts = Counter()
for row in movies_df.select('genres').collect():
    for g in row.genres.split('|'):
        if g != '(no genres listed)': genre_counts[g] += 1
top_genres = pd.DataFrame(genre_counts.most_common(10),
                           columns=['Genre','Count'])
ax5.barh(top_genres['Genre'][::-1], top_genres['Count'][::-1],
         color=sns.color_palette('magma', 10))
ax5.set_xlabel('Number of Movies'); ax5.set_title('Top 10 Genres')

plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA complete — plots saved to eda_plots.png')

NameError: name 'ratings_df' is not defined

---
## 🔧 Step 5 — Feature Engineering & Preprocessing

In [7]:
from pyspark.sql.functions import (
    col, regexp_replace, lower, concat_ws,
    collect_list, from_unixtime, year, count, avg, when
)

# ── 1. Clean ratings ──────────────────────────────────────────────────────
ratings_clean = (
    ratings_df
    .dropna()
    .withColumn('userId',  col('userId').cast(IntegerType()))
    .withColumn('movieId', col('movieId').cast(IntegerType()))
    .withColumn('rating',  col('rating').cast(FloatType()))
    .withColumn('year',    year(from_unixtime(col('timestamp'))))
    .drop('timestamp')
)

# ── 2. Extract year from movie title ─────────────────────────────────────
from pyspark.sql.functions import regexp_extract, trim
movies_clean = (
    movies_df
    .withColumn('release_year',
        regexp_extract(col('title'), r'\((\d{4})\)', 1).cast(IntegerType()))
    .withColumn('clean_title',
        trim(regexp_replace(col('title'), r'\s*\(\d{4}\)', '')))
)

# ── 3. Movie stats (popularity metrics) ──────────────────────────────────
movie_stats = (
    ratings_clean
    .groupBy('movieId')
    .agg(
        count('rating').alias('rating_count'),
        avg('rating').alias('avg_rating')
    )
    .withColumn('popularity_score',
        col('rating_count') * col('avg_rating'))
)

# ── 4. Sparsity check ─────────────────────────────────────────────────────
n_users  = ratings_clean.select('userId').distinct().count()
n_movies = ratings_clean.select('movieId').distinct().count()
n_ratings = ratings_clean.count()
sparsity = 1.0 - (n_ratings / (n_users * n_movies))

print('📊 Preprocessing complete')
print(f'   Users:    {n_users:,}')
print(f'   Movies:   {n_movies:,}')
print(f'   Ratings:  {n_ratings:,}')
print(f'   Sparsity: {sparsity:.4%}')  # Expected ~98-99%

# Train / Validation / Test split
train_df, val_df, test_df = ratings_clean.randomSplit([0.7, 0.15, 0.15], seed=42)
print(f'\n   Train:  {train_df.count():,}')
print(f'   Val:    {val_df.count():,}')
print(f'   Test:   {test_df.count():,}')

ModuleNotFoundError: No module named 'pyspark'

---
## 🤝 Step 6 — ALS Collaborative Filtering

In [8]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# ── Hyperparameter grid search ─────────────────────────────────────────────
param_grid = [
    {'rank': 10,  'regParam': 0.1,  'maxIter': 10},
    {'rank': 50,  'regParam': 0.1,  'maxIter': 15},
    {'rank': 100, 'regParam': 0.05, 'maxIter': 20},
]

evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='rating',
    predictionCol='prediction'
)
evaluator_mae = RegressionEvaluator(
    metricName='mae',
    labelCol='rating',
    predictionCol='prediction'
)

results = []
best_model, best_rmse = None, float('inf')

for params in param_grid:
    als = ALS(
        userCol='userId',
        itemCol='movieId',
        ratingCol='rating',
        coldStartStrategy='drop',
        nonnegative=True,
        **params
    )
    model = als.fit(train_df)
    preds = model.transform(val_df).dropna()
    rmse  = evaluator.evaluate(preds)
    mae   = evaluator_mae.evaluate(preds)
    results.append({**params, 'RMSE': round(rmse, 4), 'MAE': round(mae, 4)})

    if rmse < best_rmse:
        best_rmse, best_model = rmse, model

    print(f'  rank={params["rank"]:3d}  reg={params["regParam"]}  '
          f'iter={params["maxIter"]:2d}  →  RMSE={rmse:.4f}  MAE={mae:.4f}')

print(f'\n✅ Best RMSE: {best_rmse:.4f}')
print(pd.DataFrame(results).to_string(index=False))

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
# ── Final evaluation on held-out test set ────────────────────────────────
test_preds = best_model.transform(test_df).dropna()
test_rmse  = evaluator.evaluate(test_preds)
test_mae   = evaluator_mae.evaluate(test_preds)

print('🧪 Test Set Performance (Best ALS Model)')
print(f'   RMSE : {test_rmse:.4f}')
print(f'   MAE  : {test_mae:.4f}')

# ── Get top-10 recs for all users ────────────────────────────────────────
user_recs = best_model.recommendForAllUsers(10)

from pyspark.sql.functions import explode
recs_flat = (
    user_recs
    .select('userId', explode('recommendations').alias('rec'))
    .select('userId',
            col('rec.movieId').alias('movieId'),
            col('rec.rating').alias('pred_rating'))
    .join(movies_clean.select('movieId','clean_title','genres'), 'movieId')
)

print('\n📋 Sample Recommendations (User 1):')
recs_flat.filter(col('userId') == 1).show(10, truncate=False)

---
## 📝 Step 7 — Content-Based Filtering (TF-IDF)

In [ ]:
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover,
    HashingTF, IDF, Normalizer
)
from pyspark.ml import Pipeline
import numpy as np

# ── Build content corpus: title + genres + tags ───────────────────────────
tags_agg = (
    tags_df
    .groupBy('movieId')
    .agg(concat_ws(' ', collect_list('tag')).alias('tags_text'))
)

content_df = (
    movies_clean
    .join(tags_agg, 'movieId', 'left')
    .withColumn('content',
        lower(regexp_replace(
            concat_ws(' ',
                col('clean_title'),
                regexp_replace(col('genres'), '\\|', ' '),
                when(col('tags_text').isNotNull(), col('tags_text')).otherwise(F.lit(''))),
            '[^a-z0-9 ]', '')))
    .select('movieId', 'clean_title', 'genres', 'content')
    .na.drop(subset=['content'])
)

# ── TF-IDF pipeline ──────────────────────────────────────────────────────
tfidf_pipeline = Pipeline(stages=[
    Tokenizer(inputCol='content', outputCol='words'),
    StopWordsRemover(inputCol='words', outputCol='filtered'),
    HashingTF(inputCol='filtered', outputCol='tf',  numFeatures=10000),
    IDF(inputCol='tf',            outputCol='idf'),
    Normalizer(inputCol='idf',   outputCol='features', p=2.0),
])

tfidf_model   = tfidf_pipeline.fit(content_df)
content_vecs  = tfidf_model.transform(content_df)

print(f'✅ Content vectors built for {content_vecs.count():,} movies')
content_vecs.select('movieId','clean_title','genres').show(5, truncate=False)

In [ ]:
# ── Cosine Similarity function ────────────────────────────────────────────
from pyspark.sql.types import FloatType
from pyspark.sql.functions import udf

def find_similar_movies(movie_title_query: str, top_k: int = 10):
    """Given a movie title substring, return top-K similar movies by TF-IDF cosine sim."""
    # Find the target movie
    target_row = (
        content_vecs
        .filter(lower(col('clean_title')).contains(movie_title_query.lower()))
        .orderBy(col('clean_title'))
        .first()
    )
    if target_row is None:
        print(f'❌ Movie not found: {movie_title_query}')
        return None

    target_vec = np.array(target_row['features'].toArray())
    target_id  = target_row['movieId']
    print(f'🎬 Query movie: "{target_row["clean_title"]}" (genres: {target_row["genres"]})')

    @udf(FloatType())
    def cosine_sim_udf(v):
        arr  = v.toArray()
        denom = np.linalg.norm(arr) * np.linalg.norm(target_vec)
        return float(np.dot(arr, target_vec) / denom) if denom > 0 else 0.0

    similar = (
        content_vecs
        .filter(col('movieId') != target_id)
        .withColumn('similarity', cosine_sim_udf(col('features')))
        .orderBy(col('similarity').desc())
        .limit(top_k)
        .select('movieId', 'clean_title', 'genres', 'similarity')
    )
    return similar

# Test with Toy Story
similar = find_similar_movies('Toy Story', top_k=10)
if similar:
    similar.show(10, truncate=False)

---
## 🔀 Step 8 — Hybrid Ensemble Model

In [ ]:
from pyspark.sql.functions import coalesce, lit

def minmax_normalize(df, col_name):
    """Min-max scale a column to [0, 1]."""
    stats = df.agg(F.min(col_name).alias('mn'), F.max(col_name).alias('mx')).first()
    mn, mx = stats['mn'], stats['mx']
    return df.withColumn(col_name, (col(col_name) - mn) / (mx - mn + 1e-9))


def hybrid_recommend(user_id: int, alpha: float = 0.6, top_n: int = 15):
    """
    Hybrid = alpha * ALS_score + (1-alpha) * content_score
    """
    beta = 1.0 - alpha

    # -- ALS scores for this user
    als_recs = (
        recs_flat
        .filter(col('userId') == user_id)
        .select('movieId', col('pred_rating').alias('als_score'))
    )

    # -- Content profile: avg vector of movies rated ≥ 4.0
    liked_ids = (
        ratings_clean
        .filter((col('userId') == user_id) & (col('rating') >= 4.0))
        .select('movieId')
    )

    liked_vecs_rdd = (
        content_vecs
        .join(liked_ids, 'movieId')
        .select('features')
        .rdd
        .map(lambda r: r.features.toArray())
    )

    if liked_vecs_rdd.isEmpty():
        profile = np.zeros(10000)
    else:
        profile = np.array(liked_vecs_rdd.mean())

    @udf(FloatType())
    def profile_sim_udf(v):
        arr   = v.toArray()
        denom = np.linalg.norm(arr) * np.linalg.norm(profile)
        return float(np.dot(arr, profile) / denom) if denom > 0 else 0.0

    # Exclude already-watched movies
    watched = ratings_clean.filter(col('userId') == user_id).select('movieId')

    content_recs = (
        content_vecs
        .join(watched, 'movieId', 'left_anti')  # exclude watched
        .withColumn('content_score', profile_sim_udf(col('features')))
        .select('movieId', 'content_score')
    )

    # Normalize both to [0,1]
    als_norm     = minmax_normalize(als_recs,     'als_score')
    content_norm = minmax_normalize(content_recs, 'content_score')

    # Merge and compute hybrid score
    hybrid = (
        als_norm
        .join(content_norm, 'movieId', 'outer')
        .withColumn('als_score',     coalesce(col('als_score'),     lit(0.0)))
        .withColumn('content_score', coalesce(col('content_score'), lit(0.0)))
        .withColumn('hybrid_score',  col('als_score')*alpha + col('content_score')*beta)
        .join(movies_clean.select('movieId','clean_title','genres'), 'movieId')
        .orderBy(col('hybrid_score').desc())
        .limit(top_n)
        .select('clean_title', 'genres', 'hybrid_score', 'als_score', 'content_score')
    )
    return hybrid


# Demo
print('🔀 Hybrid Recommendations for User 42 (alpha=0.6)')
result = hybrid_recommend(user_id=42, alpha=0.6, top_n=10)
result.show(10, truncate=40)

---
## 📏 Step 9 — Evaluation: Precision@K, Recall@K, NDCG@K

In [ ]:
from pyspark.mllib.evaluation import RankingMetrics
from pyspark.sql.functions import collect_list, struct, sort_array, desc

# ── Build ground truth: items each user rated >= 4 in test set ────────────
ground_truth = (
    test_df
    .filter(col('rating') >= 4.0)
    .groupBy('userId')
    .agg(collect_list('movieId').alias('relevant_items'))
)

# ── Build predicted ranking from ALS ─────────────────────────────────────
predicted = (
    user_recs
    .select('userId', explode('recommendations').alias('rec'))
    .select('userId',
            col('rec.movieId').alias('movieId'),
            col('rec.rating').alias('pred_score'))
    .orderBy('userId', col('pred_score').desc())
    .groupBy('userId')
    .agg(collect_list('movieId').alias('predicted_items'))
)

# ── Join and compute ranking metrics ─────────────────────────────────────
combined = predicted.join(ground_truth, 'userId').dropna()

rdd_pairs = combined.rdd.map(
    lambda row: (row.predicted_items, row.relevant_items)
)

metrics = RankingMetrics(rdd_pairs)

print('📏 Ranking Metrics (ALS Top-10)')
print('='*40)
for k in [5, 10]:
    print(f'  Precision@{k}: {metrics.precisionAt(k):.4f}')
    print(f'  Recall@{k}:    {metrics.recallAt(k):.4f}')
    print(f'  NDCG@{k}:      {metrics.ndcgAt(k):.4f}')
    print()
print(f'  Mean Avg Precision: {metrics.meanAveragePrecision:.4f}')

In [ ]:
# ── Visual comparison of hyperparameter runs ───────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

results_df = pd.DataFrame(results)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['RMSE by Rank', 'MAE by Rank']
)

fig.add_trace(go.Scatter(
    x=results_df['rank'], y=results_df['RMSE'],
    mode='lines+markers+text',
    text=results_df['RMSE'].round(4).astype(str),
    textposition='top center',
    name='RMSE', marker=dict(size=12, color='steelblue')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=results_df['rank'], y=results_df['MAE'],
    mode='lines+markers+text',
    text=results_df['MAE'].round(4).astype(str),
    textposition='top center',
    name='MAE', marker=dict(size=12, color='coral')
), row=1, col=2)

fig.update_layout(
    title='ALS Hyperparameter Tuning Results',
    height=400,
    showlegend=True
)
fig.show()

---
## 📊 Step 10 — Interactive Dashboard (Plotly)

In [ ]:
# ── Pre-compute dashboard data ────────────────────────────────────────────
print('⏳ Computing dashboard data...')

# 1. Top-20 most popular movies
top_movies_pd = (
    movie_stats
    .join(movies_clean.select('movieId','clean_title','genres'), 'movieId')
    .filter(col('rating_count') >= 50)
    .orderBy(col('avg_rating').desc(), col('rating_count').desc())
    .limit(20)
    .select('clean_title','genres','rating_count','avg_rating','popularity_score')
    .toPandas()
)

# 2. Genre vs avg rating
from pyspark.sql.functions import split, array_join
genre_ratings_pd = (
    movies_clean
    .join(movie_stats, 'movieId')
    .select('movieId', explode(F.split(col('genres'), '\\|')).alias('genre'),
            'avg_rating', 'rating_count')
    .filter(col('genre') != '(no genres listed)')
    .groupBy('genre')
    .agg(
        F.avg('avg_rating').alias('mean_rating'),
        F.sum('rating_count').alias('total_ratings')
    )
    .orderBy('mean_rating', ascending=False)
    .toPandas()
)

# 3. User rating frequency
user_freq_pd = (
    ratings_clean
    .groupBy('userId')
    .agg(F.count('rating').alias('num_ratings'),
         F.avg('rating').alias('avg_rating'))
    .toPandas()
)

print('✅ Dashboard data ready!')

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Chart 1: Top 20 Movies Bubble Chart ──────────────────────────────────
fig1 = px.scatter(
    top_movies_pd,
    x='rating_count',
    y='avg_rating',
    size='popularity_score',
    color='avg_rating',
    hover_name='clean_title',
    hover_data={'genres': True, 'rating_count': True, 'avg_rating': ':.2f'},
    color_continuous_scale='RdYlGn',
    size_max=50,
    title='🎬 Top 20 Movies — Popularity vs. Rating Quality'
)
fig1.update_layout(height=500)
fig1.show()

In [ ]:
# ── Chart 2: Genre Analysis ───────────────────────────────────────────────
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Average Rating by Genre', 'Total Ratings by Genre']
)

fig2.add_trace(go.Bar(
    y=genre_ratings_pd['genre'],
    x=genre_ratings_pd['mean_rating'],
    orientation='h',
    marker_color=genre_ratings_pd['mean_rating'],
    marker_colorscale='Viridis',
    name='Avg Rating'
), row=1, col=1)

fig2.add_trace(go.Bar(
    y=genre_ratings_pd['genre'],
    x=genre_ratings_pd['total_ratings'],
    orientation='h',
    marker_color='steelblue',
    name='Total Ratings'
), row=1, col=2)

fig2.update_layout(title='📊 Genre Analysis', height=500, showlegend=False)
fig2.show()

In [ ]:
# ── Chart 3: Interactive Recommendation Explorer ──────────────────────────
from ipywidgets import interact, IntSlider, FloatSlider, Output
import ipywidgets as widgets

out = Output()

def show_recommendations(user_id=42, alpha=0.6, top_n=10):
    with out:
        out.clear_output(wait=True)
        recs_pd = hybrid_recommend(user_id, alpha, top_n).toPandas()

        fig = go.Figure(go.Bar(
            x=recs_pd['hybrid_score'].round(4),
            y=recs_pd['clean_title'],
            orientation='h',
            marker=dict(
                color=recs_pd['hybrid_score'],
                colorscale='Plasma',
                showscale=True
            ),
            customdata=recs_pd[['genres','als_score','content_score']],
            hovertemplate=(
                '<b>%{y}</b><br>'
                'Genres: %{customdata[0]}<br>'
                'Hybrid Score: %{x:.4f}<br>'
                'ALS Score: %{customdata[1]:.4f}<br>'
                'Content Score: %{customdata[2]:.4f}<extra></extra>'
            )
        ))
        fig.update_layout(
            title=f'🎬 Top-{top_n} Recommendations for User {user_id} (α={alpha})',
            xaxis_title='Hybrid Score',
            yaxis=dict(autorange='reversed'),
            height=max(400, top_n * 38),
            margin=dict(l=200)
        )
        fig.show()

interact(
    show_recommendations,
    user_id=IntSlider(min=1, max=610, step=1, value=42, description='User ID'),
    alpha=FloatSlider(min=0.0, max=1.0, step=0.1, value=0.6, description='ALS Weight α'),
    top_n=IntSlider(min=5, max=25, step=1, value=10, description='Top N')
)
out

In [ ]:
# ── Chart 4: User-Movie Rating Heatmap (sample) ───────────────────────────
sample_users  = [1,2,3,5,7,10,15,20,30,42]
top_movie_ids = top_movies_pd['clean_title'].head(12).tolist()

heatmap_data = pd.DataFrame(index=sample_users, columns=top_movie_ids, dtype=float)

for uid in sample_users:
    user_rows = (
        ratings_clean
        .filter(col('userId') == uid)
        .join(movies_clean.select('movieId','clean_title'), 'movieId')
        .select('clean_title','rating')
        .toPandas()
    )
    for _, row in user_rows.iterrows():
        if row['clean_title'] in heatmap_data.columns:
            heatmap_data.loc[uid, row['clean_title']] = row['rating']

fig4 = go.Figure(go.Heatmap(
    z=heatmap_data.values,
    x=[t[:30] for t in heatmap_data.columns],
    y=[f'User {u}' for u in heatmap_data.index],
    colorscale='RdYlGn',
    zmin=0.5, zmax=5.0,
    colorbar=dict(title='Rating'),
    text=heatmap_data.values,
    hoverongaps=False
))
fig4.update_layout(
    title='🔥 User × Movie Rating Heatmap',
    xaxis=dict(tickangle=-40),
    height=450
)
fig4.show()

In [ ]:
# ── Summary KPI Dashboard ────────────────────────────────────────────────
fig5 = make_subplots(
    rows=2, cols=3,
    specs=[[{'type':'indicator'}, {'type':'indicator'}, {'type':'indicator'}],
           [{'type':'scatter', 'colspan':3}, None, None]],
    subplot_titles=['Total Users','Total Movies','Best RMSE',
                    'User Activity Distribution']
)

fig5.add_trace(go.Indicator(
    mode='number+delta', value=n_users,
    delta={'reference': 500, 'relative': True},
    title={'text': 'Total Users'}
), row=1, col=1)

fig5.add_trace(go.Indicator(
    mode='number', value=n_movies,
    title={'text': 'Total Movies'}
), row=1, col=2)

fig5.add_trace(go.Indicator(
    mode='number+delta', value=round(best_rmse, 4),
    delta={'reference': 1.0, 'decreasing': {'color': 'green'}},
    title={'text': 'Best RMSE'}
), row=1, col=3)

ua_sorted = user_freq_pd.sort_values('num_ratings')
fig5.add_trace(go.Scatter(
    x=list(range(len(ua_sorted))),
    y=ua_sorted['num_ratings'],
    fill='tozeroy',
    line=dict(color='steelblue'),
    name='Ratings per user'
), row=2, col=1)

fig5.update_layout(
    title='📊 Movie Recommendation System — KPI Summary',
    height=600
)
fig5.show()
print('🎉 Dashboard complete!')

---
## 🧹 Step 11 — Save Results & Stop Spark

In [ ]:
import os
os.makedirs('output', exist_ok=True)

# Save top movie recommendations to CSV
top_movies_pd.to_csv('output/top_movies.csv', index=False)
genre_ratings_pd.to_csv('output/genre_stats.csv', index=False)
pd.DataFrame(results).to_csv('output/als_tuning_results.csv', index=False)

print('✅ Results saved to output/')
print('   output/top_movies.csv')
print('   output/genre_stats.csv')
print('   output/als_tuning_results.csv')

# Stop Spark
spark.stop()
print('\n✅ Spark session stopped. All done! 🎬')